# Settlement Attribution and Visit Reconstruction

This notebook applies the approved, non-destructive attribution scenarios in PostGIS and exports reconciled settlement-level outputs. Raw GPS observations are never updated or removed.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

import pandas as pd
from src.attribution.settlement_attribution import (SCENARIOS, create_attribution_schema, run_settlement_attribution)
from src.attribution.visit_classification import classify_settlement_visits
from src.ingestion.db_connection import get_connection
from src.project_paths import output_table_path

Project root: C:\Users\umary\Desktop\EHA test\Q1_Campaign_Team_Tracking


## Confirm database-resident reference inputs

Reference data must be loaded before this notebook is run. From the Q1 root, use `python -m src.ingestion.ingest_reference_data --source-dir <supplied_Q1_directory>`. This notebook reads database-resident inputs only.

In [2]:
REFERENCE_PRECHECK_SQL = '''
SELECT
  (SELECT COUNT(*) FROM raw.settlements) AS settlements,
  (SELECT COUNT(*) FROM reference.wards) AS wards,
  (SELECT COUNT(*) FROM reference.lgas) AS lgas,
  (SELECT COUNT(*) FROM reference.states) AS states
'''

with get_connection() as connection:
    reference_counts = pd.read_sql_query(REFERENCE_PRECHECK_SQL, connection).iloc[0].to_dict()

expected_counts = {'settlements': 2562, 'wards': 40, 'lgas': 4, 'states': 1}
if reference_counts != expected_counts:
    raise RuntimeError(
        'Reference data is absent or incomplete. Run the dedicated reference loader first. '
        f'Expected {expected_counts}; found {reference_counts}.'
    )

with get_connection() as connection:
    create_attribution_schema(connection)
    for scenario in SCENARIOS:
        run_settlement_attribution(connection, scenario)
        classify_settlement_visits(connection, scenario.name)

print('Database precheck passed; attribution scenarios and visit episodes completed.')

Database precheck passed; attribution scenarios and visit episodes completed.


## Validate and export scenario outputs

The checks below confirm one attribution per GPS point per scenario, one final settlement summary per settlement/scenario, valid settlement references, and reconciliation of exported summaries.

In [3]:
SUMMARY_SQL = '''
SELECT scenario_name, visit_classification, COUNT(*) AS settlement_count
FROM processed.settlement_visit_summaries
GROUP BY scenario_name, visit_classification
ORDER BY scenario_name, visit_classification
'''
VALIDATION_SQL = '''
SELECT scenario_name, COUNT(*) AS attributed_points, COUNT(DISTINCT gps_track_point_id) AS distinct_points,
       COUNT(*) FILTER (WHERE confidence_class = 'ambiguous') AS ambiguous_attributions
FROM processed.gps_settlement_attributions
GROUP BY scenario_name ORDER BY scenario_name
'''
with get_connection() as connection:
    scenario_summary = pd.read_sql_query(SUMMARY_SQL, connection)
    validation = pd.read_sql_query(VALIDATION_SQL, connection)
    settlement_visit_summary = pd.read_sql_query('''
        SELECT scenario_name, settlement_id, visit_classification, team_count, first_observed_at, last_observed_at,
               dwell_duration_minutes, eligible_gps_point_count, nearest_attribution_distance_m, confidence_class,
               ambiguous_attribution_count
        FROM processed.settlement_visit_summaries
        ORDER BY scenario_name, settlement_id
    ''', connection)

comparison = scenario_summary.pivot(index='scenario_name', columns='visit_classification', values='settlement_count').reset_index()
comparison['visited_change_from_baseline'] = comparison['visited'] - comparison.loc[comparison['scenario_name'].eq('baseline_30m'), 'visited'].iloc[0]
comparison['ambiguous_change_from_baseline'] = comparison['ambiguous'] - comparison.loc[comparison['scenario_name'].eq('baseline_30m'), 'ambiguous'].iloc[0]

validation.to_csv(output_table_path('settlement_attribution_summary.csv'), index=False)
settlement_visit_summary.to_csv(output_table_path('settlement_visit_summary.csv'), index=False)
comparison.to_csv(output_table_path('attribution_scenario_comparison.csv'), index=False)
comparison

visit_classification         scenario_name  ...  ambiguous_change_from_baseline
0                             baseline_30m  ...                               0
1                          sensitivity_60m  ...                              10
2                     urban_accuracy_aware  ...                               2

[3 rows x 6 columns]